**Resuming n fine-tuning yolov11s**

In [1]:
# 1. Install & Import
!pip install ultralytics -q
import os, cv2, shutil, yaml
from ultralytics import YOLO
from glob import glob
from tqdm import tqdm
from concurrent.futures import ProcessPoolExecutor

# --- CONFIGURATION ---
src_root = "/kaggle/input/datasets/shahjahanabdullatif/datasetcavitydetection/Cleaned_Dental_Dataset"
new_root = "/kaggle/working/Enhanced_Dental_Data"
input_weights = '/kaggle/input/datasets/shahjahanabdullatif/lastptyolov11s/last.pt'
working_weights = '/kaggle/working/fine_tune_start.pt'

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 21.2 MB/s eta 0:00:00
Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.


In [2]:
def preprocess_intraoral(img_path):
    try:
        rel_path = os.path.relpath(img_path, src_root)
        split = rel_path.split(os.sep)[0]
        fname = os.path.basename(img_path)
        out_img_dir = os.path.join(new_root, split, 'images')
        out_lbl_dir = os.path.join(new_root, split, 'labels')
        
        img = cv2.imread(img_path)
        if img is None: return
        lab = cv2.cvtColor(img, cv2.COLOR_BGR2LAB)
        l, a, b = cv2.split(lab)
        cl = cv2.createCLAHE(clipLimit=3.0, tileGridSize=(8,8)).apply(l)
        enhanced = cv2.cvtColor(cv2.merge((cl, a, b)), cv2.COLOR_LAB2BGR)
        
        cv2.imwrite(os.path.join(out_img_dir, fname), enhanced)
        label_src = os.path.join(src_root, split, 'labels', fname.rsplit('.', 1)[0] + ".txt")
        if os.path.exists(label_src):
            shutil.copy(label_src, os.path.join(out_lbl_dir, os.path.basename(label_src)))
    except: pass

# Create directory structure
for s in ['train', 'val', 'test']:
    os.makedirs(os.path.join(new_root, s, 'images'), exist_ok=True)
    os.makedirs(os.path.join(new_root, s, 'labels'), exist_ok=True)

all_images = glob(f"{src_root}/*/images/*")
print(f"🚀 Rebuilding dataset with {len(all_images)} images...")
with ProcessPoolExecutor() as executor:
    list(tqdm(executor.map(preprocess_intraoral, all_images), total=len(all_images)))

# 3. Re-create data.yaml
data_config = {
    'path': new_root,
    'train': 'train/images', 'val': 'val/images', 'test': 'test/images',
    'nc': 2, 'names': ['Cavity', 'Non_Cavity']
}
with open('/kaggle/working/data.yaml', 'w') as f:
    yaml.dump(data_config, f)

🚀 Rebuilding dataset with 33808 images...


100%|██████████| 33808/33808 [06:53<00:00, 81.76it/s]


In [3]:
# 4. Prepare Weights & Train
if os.path.exists(input_weights):
    shutil.copy(input_weights, working_weights)
    model = YOLO(working_weights)
    print("✅ Weights loaded. Starting Fine-Tuning...")
    
    model.train(
        data='/kaggle/working/data.yaml',
        epochs=50,
        imgsz=640,
        batch=-1,
        lr0=0.0005,
        box=7.5,
        cls=1.5,
        label_smoothing=0.05,
        close_mosaic=10,
        project='Dental_Mobile_AI',
        name='Phase2_Refinement'
    )
else:
    print("❌ ERROR: Could not find last.pt at the specified path.")

✅ Weights loaded. Starting Fine-Tuning...
WARNING ⚠️ 'label_smoothing' is deprecated and will be removed in the future.
Ultralytics 8.4.33 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=-1, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=1.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=False, cutmix=0.0, data=/kaggle/working/data.yaml, degrees=0.0, deterministic=True, device=None, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=False, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.0005, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=/kaggle/working/fine_tune_start.pt, momentum=0.937, mosaic